# Notebook 4: Mixture of Experts (MoE)

## Learning Objectives
- Understand MoE architecture
- Implement expert routing (top-k gating)
- Learn auxiliary-loss-free load balancing
- Understand shared vs routed experts

## Task 1: MoE Fundamentals

### HINT: MoE Concept
```
Traditional FFN:  x → [single FFN] → output
MoE:              x → [router] → select top-k experts → [combine outputs] → output
```

DeepSeek V3.2: 256 experts, 8-9 activated per token

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Expert(nn.Module):
    """Single expert FFN"""
    def __init__(self, hidden_size, intermediate_size):
        super().__init__()
        self.gate_up = nn.Linear(hidden_size, intermediate_size * 2)
        self.down = nn.Linear(intermediate_size, hidden_size)
        
    def forward(self, x):
        # SwiGLU: gate * up (element-wise)
        gate_up = self.gate_up(x)
        gate, up = gate_up.chunk(2, dim=-1)
        return self.down(F.silu(gate) * up)

# Test single expert
expert = Expert(hidden_size=512, intermediate_size=1376)
x = torch.randn(2, 8, 512)
output = expert(x)
print(f"Expert input: {x.shape}")
print(f"Expert output: {output.shape}")

## Task 2: Router and Top-K Gating

### HINT: Routing Function
```python
# Get logits for each expert
logits = self.gate(x)  # [batch, seq, num_experts]
# Select top-k
top_k_logits, top_k_indices = torch.topk(logits, k=2)
# Apply softmax to get weights
weights = F.softmax(top_k_logits, dim=-1)
```

In [ ]:
class MoELayer(nn.Module):
    """Mixture of Experts Layer"""
    def __init__(self, hidden_size, num_experts, num_active_experts, intermediate_size):
        super().__init__()
self.num_experts = num_experts
        self.num_active = num_active_experts
        
        # Router
        self.gate = nn.Linear(hidden_size, num_experts, bias=False)
        
        # Experts
        self.experts = nn.ModuleList([
            Expert(hidden_size, intermediate_size) for _ in range(num_experts)
        ])
        
        # Shared expert (always active)
        self.shared_expert = Expert(hidden_size, intermediate_size)
        
    def forward(self, x):
        batch_size, seq_len, hidden = x.shape
        original_shape = x.shape
        x_flat = x.view(-1, hidden)  # [batch*seq, hidden]
        
        # === SHARED EXPERT (always active) ===
        shared_output = self.shared_expert(x_flat)
        
        # === ROUTING ===
        # Get logits for each expert
        gate_logits = self.gate(x_flat)  # [batch*seq, num_experts]
        
        # Top-k routing
        top_k_logits, top_k_indices = torch.topk(gate_logits, self.num_active, dim=-1)
        
        # Normalize weights
        weights = F.softmax(top_k_logits, dim=-1)
        
        # Initialize output
        routed_output = torch.zeros_like(x_flat)
        
        # Process each token's selected experts
        for i in range(self.num_active):
            expert_id = top_k_indices[:, i]
            weight = weights[:, i].unsqueeze(-1)
            
            # Get this expert's output
            for exp_idx in range(self.num_experts):
                mask = (expert_id == exp_idx)
                if mask.any():
                    exp_input = x_flat[mask]
                    exp_output = self.experts[exp_idx](exp_input)
                    routed_output[mask] += exp_output * weight[mask]
        
        # === COMBINE ===
        output = shared_output + routed_output
        
        return output.view(*original_shape)

# Test MoE
moe = MoELayer(
    hidden_size=512,
    num_experts=8,
    num_active_experts=2,
    intermediate_size=1376
)
x = torch.randn(2, 8, 512)
output = moe(x)
print(f"MoE input: {x.shape}")
print(f"MoE output: {output.shape}")

## Task 3: Auxiliary-Loss-Free Load Balancing

### HINT: DeepSeek Innovation
Instead of adding a penalty term to loss, DeepSeek uses **dynamic bias**:

```python
# Add bias based on expert affinity
biased_logits = gate_logits + expert_bias
# expert_bias is adjusted during training to balance load
```

This maintains load balance without hurting main objective!

In [ ]:
class AuxiliaryLossFreeMoE(nn.Module):
    """MoE with aux-loss-free load balancing"""
    def __init__(self, hidden_size, num_experts, num_active_experts, intermediate_size):
        super().__init__()
        self.num_experts = num_experts
        self.num_active = num_active_experts
        
        self.gate = nn.Linear(hidden_size, num_experts, bias=False)
        
        # Expert affinity scores (learned)
        self.expert_affinity = nn.Parameter(torch.zeros(1, num_experts))
        
        self.experts = nn.ModuleList([
            Expert(hidden_size, intermediate_size) for _ in range(num_experts)
        ])
        self.shared_expert = Expert(hidden_size, intermediate_size)
        
    def forward(self, x):
        batch_size, seq_len, hidden = x.shape
        x_flat = x.view(-1, hidden)
        
        # === AUX-LOSS-FREE LOAD BALANCING ===
        # Add learned bias to gate logits
        gate_logits = self.gate(x_flat)
        biased_logits = gate_logits + self.expert_affinity  # Dynamic bias!
        
        # Shared expert
        shared_output = self.shared_expert(x_flat)
        
        # Top-k routing with biased logits
        top_k_logits, top_k_indices = torch.topk(biased_logits, self.num_active, dim=-1)
        weights = F.softmax(top_k_logits, dim=-1)
        
        # Route to experts
        routed_output = torch.zeros_like(x_flat)
        for i in range(self.num_active):
            expert_id = top_k_indices[:, i]
            weight = weights[:, i].unsqueeze(-1)
            for exp_idx in range(self.num_experts):
                mask = (expert_id == exp_idx)
                if mask.any():
                    exp_output = self.experts[exp_idx](x_flat[mask])
                    routed_output[mask] += exp_output * weight[mask]
        
        return (shared_output + routed_output).view(batch_size, seq_len, hidden)

# Test with load balancing
moe_balanced = AuxiliaryLossFreeMoE(
    hidden_size=512,
    num_experts=8,
    num_active_experts=2,
    intermediate_size=1376
)
print("Expert affinity (learned bias):", moe_balanced.expert_affinity)
output = moe_balanced(x)
print(f"Output: {output.shape}")

## Task 4: Expert Capacity

### HINT: Capacity Factor
Each expert has limited capacity:
```python
capacity = tokens_per_expert * capacity_factor
if token_count > capacity: overflow
```

In [ ]:
# Compare parameter counts
def count_params(module):
    return sum(p.numel() for p in module.parameters())

print("=== Parameter Comparison ===")
print(f"Dense FFN:     {count_params(Expert(512, 1376)):,}")
print(f"MoE (8 exp):   {count_params(moe):,}")
print(f"Active in MoE (2): {count_params(moe.experts[0]) * 2 + count_params(moe.shared_expert):,}")
print()
print(f"=== Memory & Compute ===")
print(f"Dense: All params active")
print(f"MoE: Only 2/8 + 1 shared experts = 3 FFNs active")
print(f"→ {(1 - 3/9)*100:.0f}% parameter reduction while maintaining capacity!")

## Verification

Complete these checks:
1. ✅ Understand MoE architecture
2. ✅ Can implement expert routing
3. ✅ Understand aux-loss-free load balancing
4. ✅ Know difference between shared and routed experts